### Bronze PipelineIngests raw source data with no transformation logic beyond capturing the data as-is.Sources: - `transactions` table via JDBC (source database) - `mcc_codes.json` from a Databricks Volume (MCC code -> description lookup) - `train_fraud_labels.json` from a Databricks Volume (transaction id -> fraud flag)This notebook is intended to be run as the first task (`Bronze`) in the pipeline job. Running the whole notebook refreshes all bronze tables.

In [0]:
%sql-- Setup catalog if not existsCREATE CATALOG IF NOT EXISTS catalog;-- Setup schemas for medallion architectureCREATE SCHEMA IF NOT EXISTS catalog.bronze;CREATE SCHEMA IF NOT EXISTS catalog.silver;CREATE SCHEMA IF NOT EXISTS catalog.gold;

In [0]:
from pyspark.sql import functions as Ffrom pyspark.sql.types import (    StructType, StructField, StringType, DoubleType, IntegerType, LongType)

In [0]:
# Insert your database credentials and URLurl = "your database url"transactions_schema = StructType([    StructField("id", LongType(), True),    StructField("date", StringType(), True),    StructField("client_id", LongType(), True),    StructField("card_id", LongType(), True),    StructField("amount", StringType(), True),      # arrives as "$123.45", cleaned in silver    StructField("use_chip", StringType(), True),    StructField("merchant_id", LongType(), True),    StructField("merchant_city", StringType(), True),    StructField("merchant_state", StringType(), True),    StructField("zip", StringType(), True),    StructField("mcc", IntegerType(), True),    StructField("errors", StringType(), True),])raw_transactions_df = (spark.read    .format("jdbc")    .option("url", url)    .option("dbtable", "transactions")    .option("user", "your_user")    .option("password", "your_password")    .load())display(raw_transactions_df)

In [0]:
# mcc_codes.json is a flat JSON object: {"5411": "Grocery Stores, Supermarkets", ...}mcc_codes_raw_df = (spark.read    .option("multiline", "true")    .json("/Volumes/catalog/bronze_landing/mcc_codes.json"))# The file loads as a single row with one column per mcc code, so we stack it into rowsmcc_codes_df = (mcc_codes_raw_df    .select(F.explode(F.map_from_arrays(        F.array([F.lit(c) for c in mcc_codes_raw_df.columns]),        F.array([F.col(c) for c in mcc_codes_raw_df.columns])    )).alias("mcc", "mcc_description")))display(mcc_codes_df)

In [0]:
# train_fraud_labels.json has the shape: {"target": {"<transaction_id>": "Yes"/"No", ...}}fraud_labels_raw_df = (spark.read    .option("multiline", "true")    .json("/Volumes/catalog/bronze_landing/train_fraud_labels.json"))target_col = fraud_labels_raw_df.select("target")target_struct = target_col.schema[0].dataTypefraud_labels_df = (fraud_labels_raw_df    .select(F.explode(F.map_from_arrays(        F.array([F.lit(f.name) for f in target_struct.fields]),        F.array([F.col(f"target.`{f.name}`") for f in target_struct.fields])    )).alias("transaction_id", "fraud_flag"))    .withColumn("transaction_id", F.col("transaction_id").cast("long")))display(fraud_labels_df)

In [0]:
raw_transactions_df.write.mode("overwrite").saveAsTable("catalog.bronze.transactions")mcc_codes_df.write.mode("overwrite").saveAsTable("catalog.bronze.mcc_codes")fraud_labels_df.write.mode("overwrite").saveAsTable("catalog.bronze.fraud_labels")